# 02 · Evaluación de la entrega

Proyecto Oráculo, ADAG1, Grupo 1.

**Cómo correrlo:** elige T4 GPU y usa **Ejecutar todo**. La primera vez el entorno se reinicia solo después de instalar; cuando reconecte, usa **Ejecutar todo** otra vez. No hay que editar nada.

Si existe `datos_privados.json` junto al notebook, evalúa esos datos. Si no existe, usa la partición de validación para comprobar que todo funciona.

In [ ]:
import os

if not os.path.exists("listo.txt"):
    # bitsandbytes y las librerías de nltk/spacy son las que pide el notebook del curso.
    !pip install -q -U "bitsandbytes>=0.46.1" transformers accelerate nltk spacy emoji langdetect immutabledict
    !python -m spacy download en_core_web_sm -q
    !test -d open-instruct || git clone -q https://github.com/allenai/open-instruct

    CURSO = "https://raw.githubusercontent.com/JuManoel/ADA-2026-2-GP01/main"
    !wget -q --no-cache -O oraculo.py {CURSO}/oraculo.py
    !wget -q --no-cache -O ayudas.py {CURSO}/ayudas.py
    !wget -q --no-cache -O datos_visibles.json {CURSO}/datos_visibles.json

    # Código y entrega del equipo.
    !test -d equipo || git clone -q --depth 1 --branch solucion-base https://github.com/samyquito/SYY-Adag1-Oraculo equipo
    !cp -r equipo/src . 2>/dev/null || true
    !test -f equipo/entrega.json && cp equipo/entrega.json . || true

    import io
    import urllib.request
    import zipfile

    NLTK_DATA = "https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages"
    for carpeta, nombre in [
        ("tokenizers", "punkt"),
        ("tokenizers", "punkt_tab"),
        ("taggers", "averaged_perceptron_tagger"),
        ("taggers", "averaged_perceptron_tagger_eng"),
    ]:
        with urllib.request.urlopen(f"{NLTK_DATA}/{carpeta}/{nombre}.zip") as resp:
            zipfile.ZipFile(io.BytesIO(resp.read())).extractall(f"/root/nltk_data/{carpeta}")
    open("listo.txt", "w").close()
    os.kill(os.getpid(), 9)

In [ ]:
import json

from ayudas import cargar_datos, cargar_modelo, dividir
from oraculo import Oraculo
from src.busqueda import validar_entrega

MODELO = "qwen17b"
RUTA_PRIVADA = "datos_privados.json"

entrega = json.load(open("entrega.json"))
validar_entrega(entrega)
config = entrega["config"]
print("grupo", entrega["grupo"], "· semana", entrega["semana"], "·", config)

if os.path.exists(RUTA_PRIVADA):
    prueba = json.load(open(RUTA_PRIVADA))
    fuente = "datos privados"
else:
    _, prueba = dividir(cargar_datos())
    fuente = "validación"

modelo = cargar_modelo(MODELO)
oraculo = Oraculo(modelo, prueba, cache=None, lote=16, presupuesto=32_000_000)
r = oraculo.evaluar(config, prueba, semilla=1)
print(f"{fuente}: precisión {r.precision:.1%} en {r.n} instancias")